# Election Prediction Project
## Notebook 05 — Hybrid Model (Demographics + Previous-Election History)

### Notebook Objectives
The demographic model (notebook 03) answers *how much of voting is explained by demographics*.
A separate question is *what is the best practical forecast* — and for that, a locality's own
previous election result is an extremely strong signal (see the persistence baseline in notebook 03).

This notebook builds a **hybrid** model that adds four lag features — each locality's four bloc
shares in the immediately preceding election — alongside the demographic features, and asks whether
demographics can improve on a pure persistence forecast.

### Design
- **Lag features:** for each locality-election row, the four bloc shares of the previous election.
- **Scope:** Knesset 22–25 only (each has a genuine predecessor). Knesset 21 has no predecessor, so
  it is used only as the lag *source* for Knesset 22 and is never a target row — avoiding any target leakage.
- **Same discipline as notebook 03:** architecture selection on Knesset 24 (training on 22–23);
  the winner is retrained on 22–24 and evaluated once on the held-out Knesset 25 test election.
- **Delta experiment:** a direct test of the project's central claim — predicting each locality's
  between-election *swing* (Δ = current − previous share) from demographics only; the expected result is R² ≈ 0.

### Input
- `data/processed/modeling_dataset_selected_features.csv`
- `data/processed/selected_demographic_features.json`
- `data/processed/election_targets_by_locality.csv` (source of the lag features)

### Output
- `reports/hybrid_model_comparison.csv`
- `reports/hybrid_summary.json`

## 1. Imports and configuration

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

RANDOM_STATE = 99
SELECTION_VALIDATION_ELECTION = "Knesset_24"
TEST_ELECTION = "Knesset_25"

TARGET_COLUMNS = ["Right_pct", "Center_Left_pct", "Haredi_pct", "Arab_pct"]
BASE_CLASSIFICATION_FEATURES = [
    "type_Arab/Non-Jewish", "type_Cities", "type_Kibbutzim",
    "type_Moshavim", "type_other",
]
DRUZE_FEATURE = "type_Druze_Majority"
LAG_FEATURES = [f"lag_{target}" for target in TARGET_COLUMNS]

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
REPO_ROOT = Path('..')

PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = REPO_ROOT / 'reports'

INPUT_PATH = PROCESSED_DIR / 'modeling_dataset_selected_features.csv'
SELECTED_FEATURES_PATH = PROCESSED_DIR / 'selected_demographic_features.json'
ELECTION_TARGETS_PATH = PROCESSED_DIR / 'election_targets_by_locality.csv'

HYBRID_COMPARISON_PATH = REPORTS_DIR / 'hybrid_model_comparison.csv'
HYBRID_SUMMARY_PATH = REPORTS_DIR / 'hybrid_summary.json'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Input dataset: {INPUT_PATH}')

Input dataset: ..\data\processed\modeling_dataset_selected_features.csv


## 2. Load data and build the lag features

In [3]:
df = pd.read_csv(INPUT_PATH, low_memory=False)

with open(SELECTED_FEATURES_PATH, 'r', encoding='utf-8') as file:
    demographic_features = json.load(file)['selected_demographic_features']

for frame_column in ['locality_symbol']:
    df[frame_column] = (
        pd.to_numeric(df[frame_column], errors='coerce')
        .astype('Int64')
        .astype('string')
    )

numeric_columns = list(dict.fromkeys(
    demographic_features + BASE_CLASSIFICATION_FEATURES
    + [DRUZE_FEATURE] + TARGET_COLUMNS
))
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# ---- lag features: each locality's bloc shares in the PREVIOUS election ----
PREVIOUS_ELECTION = {
    'Knesset_22': 'Knesset_21',
    'Knesset_23': 'Knesset_22',
    'Knesset_24': 'Knesset_23',
    'Knesset_25': 'Knesset_24',
}
# the election each source row provides lag data FOR
SERVES_AS_LAG_FOR = {previous: current for current, previous in PREVIOUS_ELECTION.items()}

all_targets = pd.read_csv(ELECTION_TARGETS_PATH)
all_targets['locality_symbol'] = (
    pd.to_numeric(all_targets['locality_symbol'], errors='coerce')
    .astype('Int64')
    .astype('string')
)

lag_source = all_targets[['locality_symbol', 'target_election'] + TARGET_COLUMNS].copy()
lag_source = lag_source.rename(columns={t: f'lag_{t}' for t in TARGET_COLUMNS})
lag_source['target_election'] = lag_source['target_election'].map(SERVES_AS_LAG_FOR)
lag_source = lag_source.dropna(subset=['target_election'])

df = df.merge(lag_source, on=['locality_symbol', 'target_election'], how='left')

# Keep only elections that have a genuine predecessor and complete lag values.
hybrid_df = df[df['target_election'].isin(PREVIOUS_ELECTION.keys())].copy()
before = len(hybrid_df)
hybrid_df = hybrid_df.dropna(subset=LAG_FEATURES)

print(f'Hybrid rows (Knesset 22-25): {len(hybrid_df)} (dropped {before - len(hybrid_df)} without complete lag)')
print('Lag features:', LAG_FEATURES)
display(hybrid_df[['locality_symbol', 'target_election'] + TARGET_COLUMNS + LAG_FEATURES].head())

Hybrid rows (Knesset 22-25): 898 (dropped 1 without complete lag)
Lag features: ['lag_Right_pct', 'lag_Center_Left_pct', 'lag_Haredi_pct', 'lag_Arab_pct']


,locality_symbol,target_election,Right_pct,Center_Left_pct,Haredi_pct,Arab_pct,lag_Right_pct,lag_Center_Left_pct,lag_Haredi_pct,lag_Arab_pct
1,1015,Knesset_22,42.739543,50.927025,6.088698,0.244735,48.198100,47.712276,3.901095,0.188529
2,1015,Knesset_23,45.330380,47.818383,6.253230,0.598007,42.739543,50.927025,6.088698,0.244735
3,1015,Knesset_24,48.337887,46.053430,5.380996,0.227687,45.330380,47.818383,6.253230,0.598007
4,1015,Knesset_25,45.163610,48.212873,6.220784,0.402733,48.337887,46.053430,5.380996,0.227687
6,1020,Knesset_22,58.102848,29.635321,12.149150,0.112682,65.181347,25.233161,9.544041,0.041451


## 3. Chronological selection / validation / test split

In [4]:
selection_validation_mask = hybrid_df['target_election'].eq(SELECTION_VALIDATION_ELECTION)
test_mask = hybrid_df['target_election'].eq(TEST_ELECTION)

selection_train_df = hybrid_df.loc[~selection_validation_mask & ~test_mask].copy()
selection_validation_df = hybrid_df.loc[selection_validation_mask].copy()
final_train_df = hybrid_df.loc[~test_mask].copy()
test_df = hybrid_df.loc[test_mask].copy()

for name, subset in {
    'selection-training (Knesset 22-23)': selection_train_df,
    'selection-validation (Knesset 24)': selection_validation_df,
    'final-training (Knesset 22-24)': final_train_df,
    'test (Knesset 25)': test_df,
}.items():
    if subset.empty:
        raise ValueError(f'The {name} subset is empty.')

y_selection_validation = selection_validation_df[TARGET_COLUMNS].copy()
y_test = test_df[TARGET_COLUMNS].copy()

print(f'Selection-training rows (Knesset 22-23): {len(selection_train_df)}')
print(f'Selection-validation rows (Knesset 24): {len(selection_validation_df)}')
print(f'Final-training rows (Knesset 22-24): {len(final_train_df)}')
print(f'Test rows (Knesset 25): {len(test_df)}')

Selection-training rows (Knesset 22-23): 449
Selection-validation rows (Knesset 24): 224
Final-training rows (Knesset 22-24): 673
Test rows (Knesset 25): 225


## 4. Reusable modeling functions

Identical to notebook 03 — imputation fitted on training data only, CLR compositional transform, and the segmented fitter.

In [5]:
def prepare_feature_matrices(
    training_data,
    validation_data,
    feature_columns
):
    """Fit median imputation on training data only."""

    usable_features = [
        column
        for column in feature_columns
        if not training_data[column].isna().all()
    ]

    if not usable_features:
        raise ValueError("No usable features remain.")

    imputer = SimpleImputer(strategy="median")

    X_train = pd.DataFrame(
        imputer.fit_transform(training_data[usable_features]),
        columns=usable_features,
        index=training_data.index
    )

    X_validation = pd.DataFrame(
        imputer.transform(validation_data[usable_features]),
        columns=usable_features,
        index=validation_data.index
    )

    return {
        "X_train": X_train,
        "X_validation": X_validation,
        "imputer": imputer,
        "features": usable_features,
    }


def evaluate_predictions(y_true, predictions, model_name):
    """Return overall and target-level MAE and R2."""

    y_true_array = np.asarray(y_true, dtype=float)
    predictions = np.asarray(predictions, dtype=float)

    overall_result = {
        "Model": model_name,
        "Overall_MAE": float(
            mean_absolute_error(y_true_array, predictions)
        ),
        "Overall_R2": float(
            r2_score(y_true_array, predictions)
        ),
    }

    target_mae = mean_absolute_error(
        y_true_array,
        predictions,
        multioutput="raw_values"
    )

    target_r2 = r2_score(
        y_true_array,
        predictions,
        multioutput="raw_values"
    )

    target_rows = [
        {
            "Model": model_name,
            "Target": target,
            "MAE": float(mae_value),
            "R2": float(r2_value),
        }
        for target, mae_value, r2_value in zip(
            TARGET_COLUMNS,
            target_mae,
            target_r2
        )
    ]

    return overall_result, target_rows


def normalize_predictions(predictions):
    """Clip negative values and normalize each row to 100%."""

    clipped = np.clip(
        np.asarray(predictions, dtype=float),
        0,
        None
    )

    row_sums = clipped.sum(axis=1, keepdims=True)

    zero_rows = row_sums.squeeze() == 0

    if zero_rows.any():
        clipped[zero_rows] = 1.0
        row_sums = clipped.sum(axis=1, keepdims=True)

    return clipped / row_sums * 100.0


def clr_transform(percentages, epsilon=1e-4):
    """Transform percentage compositions into CLR values."""

    compositions = np.asarray(percentages, dtype=float) / 100.0
    compositions = np.clip(compositions, epsilon, None)
    compositions = compositions / compositions.sum(
        axis=1,
        keepdims=True
    )

    log_values = np.log(compositions)

    return log_values - log_values.mean(
        axis=1,
        keepdims=True
    )


def inverse_clr(clr_values):
    """Convert CLR values into positive percentages summing to 100%."""

    clr_values = np.asarray(clr_values, dtype=float)
    stabilized = clr_values - clr_values.max(
        axis=1,
        keepdims=True
    )

    exponentials = np.exp(stabilized)

    return (
        exponentials
        / exponentials.sum(axis=1, keepdims=True)
        * 100.0
    )

In [6]:
def make_xgb_regressor():
    """Return the common XGBoost configuration."""

    return XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        objective="reg:squarederror",
        n_jobs=-1,
    )


def fit_independent_model(
    training_data,
    validation_data,
    feature_columns,
    y_training,
    estimator
):
    """Train independent regressors on raw percentage targets."""

    prepared = prepare_feature_matrices(
        training_data,
        validation_data,
        feature_columns
    )

    model = MultiOutputRegressor(clone(estimator))

    model.fit(prepared["X_train"], y_training)

    predictions = model.predict(
        prepared["X_validation"]
    )

    return {
        **prepared,
        "model": model,
        "predictions": predictions,
    }


def fit_clr_model(
    training_data,
    validation_data,
    feature_columns,
    y_training
):
    """Train XGBoost models on CLR-transformed targets."""

    prepared = prepare_feature_matrices(
        training_data,
        validation_data,
        feature_columns
    )

    y_training_clr = clr_transform(y_training)

    model = MultiOutputRegressor(
        make_xgb_regressor()
    )

    model.fit(
        prepared["X_train"],
        y_training_clr
    )

    predicted_clr = model.predict(
        prepared["X_validation"]
    )

    predictions = inverse_clr(predicted_clr)

    return {
        **prepared,
        "model": model,
        "predictions": predictions,
        "predicted_clr": predicted_clr,
    }


def fit_segmented_clr_model(
    training_data,
    validation_data,
    arab_feature_columns,
    non_arab_feature_columns
):
    """Train separate CLR models for Arab and non-Arab localities."""

    group_column = "type_Arab/Non-Jewish"

    arab_train = training_data.loc[
        training_data[group_column].eq(1)
    ].copy()

    arab_validation = validation_data.loc[
        validation_data[group_column].eq(1)
    ].copy()

    non_arab_train = training_data.loc[
        training_data[group_column].eq(0)
    ].copy()

    non_arab_validation = validation_data.loc[
        validation_data[group_column].eq(0)
    ].copy()

    for name, subset in {
        "Arab training": arab_train,
        "Arab validation": arab_validation,
        "Non-Arab training": non_arab_train,
        "Non-Arab validation": non_arab_validation,
    }.items():
        if subset.empty:
            raise ValueError(f"{name} subset is empty.")

    arab_result = fit_clr_model(
        arab_train,
        arab_validation,
        arab_feature_columns,
        arab_train[TARGET_COLUMNS]
    )

    non_arab_result = fit_clr_model(
        non_arab_train,
        non_arab_validation,
        non_arab_feature_columns,
        non_arab_train[TARGET_COLUMNS]
    )

    predictions_df = pd.DataFrame(
        index=validation_data.index,
        columns=TARGET_COLUMNS,
        dtype=float
    )

    predictions_df.loc[
        arab_validation.index,
        TARGET_COLUMNS
    ] = arab_result["predictions"]

    predictions_df.loc[
        non_arab_validation.index,
        TARGET_COLUMNS
    ] = non_arab_result["predictions"]

    if predictions_df.isna().any().any():
        raise ValueError(
            "Some validation rows did not receive predictions."
        )

    return {
        "predictions": (
            predictions_df
            .loc[validation_data.index]
            .to_numpy(dtype=float)
        ),
        "arab_result": arab_result,
        "non_arab_result": non_arab_result,
        "arab_train_rows": len(arab_train),
        "arab_validation_rows": len(arab_validation),
        "non_arab_train_rows": len(non_arab_train),
        "non_arab_validation_rows": len(non_arab_validation),
    }

## 5. Hybrid architecture selection (validated on Knesset 24)

Each architecture is trained with the demographic features **plus** the four lag features. A
lag-only reference (the four lag features through a model, no demographics) shows what pure history
contributes. Selection uses Knesset 24 only.

In [7]:
demographic_and_lag = list(dict.fromkeys(demographic_features + LAG_FEATURES))

non_arab_base = [c for c in BASE_CLASSIFICATION_FEATURES if c != 'type_Arab/Non-Jewish']
arab_hybrid_features = list(dict.fromkeys(demographic_features + LAG_FEATURES))
non_arab_hybrid_features = list(dict.fromkeys(demographic_features + non_arab_base + LAG_FEATURES))
full_hybrid_features = list(dict.fromkeys(demographic_features + BASE_CLASSIFICATION_FEATURES + LAG_FEATURES))

HYBRID_BUILDERS = {
    'Lag-only CLR XGBoost': lambda tr, ev: fit_clr_model(
        tr, ev, LAG_FEATURES, tr[TARGET_COLUMNS]),
    'Hybrid XGBoost + Types + Lag': lambda tr, ev: fit_independent_model(
        tr, ev, full_hybrid_features, tr[TARGET_COLUMNS], make_xgb_regressor()),
    'Hybrid CLR XGBoost + Lag': lambda tr, ev: fit_clr_model(
        tr, ev, full_hybrid_features, tr[TARGET_COLUMNS]),
    'Hybrid Segmented CLR + Lag': lambda tr, ev: fit_segmented_clr_model(
        tr, ev, arab_hybrid_features, non_arab_hybrid_features),
}

selection_results = []
for model_name, builder in HYBRID_BUILDERS.items():
    print(f'Evaluating on Knesset 24: {model_name}')
    result = builder(selection_train_df, selection_validation_df)
    overall_row, _ = evaluate_predictions(y_selection_validation, result['predictions'], model_name)
    selection_results.append(overall_row)
    print(f"  MAE: {overall_row['Overall_MAE']:.3f} | R2: {overall_row['Overall_R2']:.3f}")

hybrid_comparison_df = (
    pd.DataFrame(selection_results).sort_values('Overall_MAE').reset_index(drop=True)
)
display(hybrid_comparison_df.round(3))

hybrid_model_name = hybrid_comparison_df.iloc[0]['Model']
print(f'Selected hybrid architecture (lowest Knesset-24 MAE): {hybrid_model_name}')

Evaluating on Knesset 24: Lag-only CLR XGBoost


  MAE: 4.431 | R2: 0.927
Evaluating on Knesset 24: Hybrid XGBoost + Types + Lag


  MAE: 3.888 | R2: 0.943
Evaluating on Knesset 24: Hybrid CLR XGBoost + Lag


  MAE: 4.097 | R2: 0.931
Evaluating on Knesset 24: Hybrid Segmented CLR + Lag


  MAE: 4.267 | R2: 0.924


,Model,Overall_MAE,Overall_R2
0,Hybrid XGBoost + Types + Lag,3.888,0.943
1,Hybrid CLR XGBoost + Lag,4.097,0.931
2,Hybrid Segmented CLR + Lag,4.267,0.924
3,Lag-only CLR XGBoost,4.431,0.927


Selected hybrid architecture (lowest Knesset-24 MAE): Hybrid XGBoost + Types + Lag


## 6. Final hybrid model — retrained on Knesset 22–24, tested once on Knesset 25

In [8]:
final_hybrid = HYBRID_BUILDERS[hybrid_model_name](final_train_df, test_df)
hybrid_predictions = final_hybrid['predictions']

hybrid_test_overall, hybrid_test_targets = evaluate_predictions(
    y_test, hybrid_predictions, hybrid_model_name
)

hybrid_sums = hybrid_predictions.sum(axis=1)
print(f'Prediction sum range: {hybrid_sums.min():.4f} - {hybrid_sums.max():.4f}')
print(f"Hybrid test MAE (Knesset 25): {hybrid_test_overall['Overall_MAE']:.3f}")
print(f"Hybrid test R2:  {hybrid_test_overall['Overall_R2']:.3f}")
print('\nPer-bloc test MAE:')
for row in hybrid_test_targets:
    print(f"  {row['Target']:16s} {row['MAE']:.3f}")

Prediction sum range: 82.4767 - 109.4672
Hybrid test MAE (Knesset 25): 3.627
Hybrid test R2:  0.943

Per-bloc test MAE:
  Right_pct        5.387
  Center_Left_pct  3.813
  Haredi_pct       1.503
  Arab_pct         3.807


## 7. Delta experiment — can demographics predict between-election swings?

A direct empirical test of the project's central claim. The target here is not the bloc share
itself but the **change** in each bloc's share relative to the previous election
(Δ = current − lag). If demographic structure drove the swings between elections, the static
demographic features should explain part of this variance; the expectation, given everything
above, is R² ≈ 0. A "no swing" baseline (predicting zero change everywhere) is shown for scale.


In [9]:
DELTA_COLUMNS = [f'delta_{target}' for target in TARGET_COLUMNS]
for target in TARGET_COLUMNS:
    hybrid_df[f'delta_{target}'] = hybrid_df[target] - hybrid_df[f'lag_{target}']

delta_train_df = hybrid_df.loc[~hybrid_df['target_election'].eq(TEST_ELECTION)].copy()
delta_test_df = hybrid_df.loc[hybrid_df['target_election'].eq(TEST_ELECTION)].copy()

prepared_delta = prepare_feature_matrices(
    delta_train_df, delta_test_df, demographic_features
)
delta_model = MultiOutputRegressor(make_xgb_regressor())
delta_model.fit(prepared_delta['X_train'], delta_train_df[DELTA_COLUMNS])
delta_predictions = delta_model.predict(prepared_delta['X_validation'])

delta_actual = delta_test_df[DELTA_COLUMNS].to_numpy(dtype=float)
delta_r2 = float(r2_score(delta_actual, delta_predictions))
delta_mae = float(mean_absolute_error(delta_actual, delta_predictions))
delta_zero_baseline_mae = float(
    mean_absolute_error(delta_actual, np.zeros_like(delta_actual))
)

print(f'Delta-experiment training rows (Knesset 22-24): {len(delta_train_df)}')
print(f'Delta-experiment test rows (Knesset 25): {len(delta_test_df)}')
print(f'R2 of demographics on the swings: {delta_r2:.3f}')
print(f'MAE of demographics on the swings: {delta_mae:.3f}')
print(f'MAE of the "no swing" baseline (all zeros): {delta_zero_baseline_mae:.3f}')


Delta-experiment training rows (Knesset 22-24): 673
Delta-experiment test rows (Knesset 25): 225
R2 of demographics on the swings: -0.971
MAE of demographics on the swings: 3.827
MAE of the "no swing" baseline (all zeros): 3.228


## 8. The three questions side by side

In [10]:
demographic_summary = json.load(open(REPORTS_DIR / 'modeling_summary.json', encoding='utf-8'))

comparison = pd.DataFrame([
    {'Approach': 'Baseline: Persistence (previous election)',
     'Uses': 'election history only',
     'Test MAE': demographic_summary['baseline_persistence_test_mae']},
    {'Approach': 'Baseline: Persistence (2-election mean)',
     'Uses': 'election history only',
     'Test MAE': demographic_summary['baseline_persistence_2mean_test_mae']},
    {'Approach': f"Demographic model ({demographic_summary['selected_model']})",
     'Uses': 'demographics only',
     'Test MAE': demographic_summary['overall_mae']},
    {'Approach': f'Hybrid model ({hybrid_model_name})',
     'Uses': 'demographics + history',
     'Test MAE': hybrid_test_overall['Overall_MAE']},
]).sort_values('Test MAE').reset_index(drop=True)

display(comparison.round(3))

hybrid_comparison_df.to_csv(HYBRID_COMPARISON_PATH, index=False, encoding='utf-8-sig')

hybrid_summary = {
    'selected_hybrid_model': hybrid_model_name,
    'selection_validation_election': SELECTION_VALIDATION_ELECTION,
    'test_election': TEST_ELECTION,
    'hybrid_feature_count': int(len(full_hybrid_features)),
    'lag_features': LAG_FEATURES,
    'hybrid_test_mae': float(hybrid_test_overall['Overall_MAE']),
    'hybrid_test_r2': float(hybrid_test_overall['Overall_R2']),
    'demographic_test_mae': float(demographic_summary['overall_mae']),
    'baseline_persistence_test_mae': float(demographic_summary['baseline_persistence_test_mae']),
    'baseline_persistence_2mean_test_mae': float(demographic_summary['baseline_persistence_2mean_test_mae']),
    'hybrid_test_mae_by_target': {row['Target']: float(row['MAE']) for row in hybrid_test_targets},
    'delta_experiment_r2': float(delta_r2),
    'delta_experiment_mae': float(delta_mae),
    'delta_zero_baseline_mae': float(delta_zero_baseline_mae),
}
with open(HYBRID_SUMMARY_PATH, 'w', encoding='utf-8') as file:
    json.dump(hybrid_summary, file, ensure_ascii=False, indent=2)

print(f'Hybrid artifacts saved:\n- {HYBRID_COMPARISON_PATH}\n- {HYBRID_SUMMARY_PATH}')

,Approach,Uses,Test MAE
0,Baseline: Persistence (2-election mean),election history only,2.273
1,Baseline: Persistence (previous election),election history only,3.296
2,Hybrid model (Hybrid XGBoost + Types + Lag),demographics + history,3.627
3,Demographic model (Segmented CLR),demographics only,4.207


Hybrid artifacts saved:
- ..\reports\hybrid_model_comparison.csv
- ..\reports\hybrid_summary.json


## 9. Notebook summary

Three approaches answer three different questions:

- **Persistence baseline** — "the locality votes like last time." Strong on raw accuracy but explains
  nothing and cannot handle a locality without a comparable prior.
- **Demographic model** — "how much does *who lives here* explain the vote?" A scientific measure
  (R² ≈ 0.92 from demographics alone), and the only approach that generalizes to demographic-shift
  scenarios and driver analysis.
- **Hybrid model** — "best practical forecast" — combines a locality's history with its demographics;
  demographics correct the drift of a pure persistence forecast.

The comparison table above places all three (and the persistence baselines) on the same held-out
Knesset 25 test election.
The delta experiment closes the argument empirically: predicting the between-election swing itself from demographics yields R² ≈ 0 — demographics explain the *structure* of the political map, while the swings between elections are driven by politics.
